# Q2 Linear Regression Diagnostics (Linear Probability Model)

This notebook is the **linear-model companion** to `q2_diagnostics.ipynb`.

What this file does:
- loads `../data/sample_q2.parquet` using the same core feature set,
- fits a linear regression model (`statsmodels.OLS`) for `booking_bool`,
- compares performance against a simple baseline,
- visualizes residual behavior and summarizes coefficient effects.

How to use it:
1. Run cells top-to-bottom.
2. Use the metrics table to compare baseline vs OLS.
3. Use coefficient and residual sections to write interpretation + caveats.

Because the target is binary, this is a **linear probability model (LPM)**. It is used for interpretability and side-by-side comparison with logistic regression, not as the final probability model.

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 1) Load Data and Define Features

We use the same core feature set as `q2_diagnostics.ipynb` so results are directly comparable.

Target is `booking_bool` (0/1), which makes this a linear probability model.

In [ ]:
DATA_PATH = '../data/sample_q2.parquet'
N_SAMPLE = 100_000  # reduce/increase as needed for speed

FEATURES = [
    'position',
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score1',
    'log_price_usd',
    'prop_log_historical_price',
    'srch_booking_window',
    'srch_length_of_stay',
    'srch_adults_count',
    'srch_saturday_night_bool',
]

TARGET = 'booking_bool'

# Keep only required columns and drop missing values for clean OLS fitting
df_full = pd.read_parquet(DATA_PATH)
required_cols = FEATURES + [TARGET]
missing_required = [c for c in required_cols if c not in df_full.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

df_clean = df_full[required_cols].dropna().copy()
if len(df_clean) > N_SAMPLE:
    df = df_clean.sample(n=N_SAMPLE, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df = df_clean.reset_index(drop=True)

print(f'Full parquet rows: {len(df_full):,}')
print(f'After dropna    : {len(df_clean):,}')
print(f'Working sample  : {len(df):,}')
print(f'Booking rate    : {df[TARGET].mean():.3%}')
df.head()

## 2) Train/Test Split and Baseline

We compare OLS against a simple baseline that predicts the training-set mean booking rate for every test observation.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Add intercept for statsmodels OLS
X_train_sm = sm.add_constant(X_train, has_constant='add')
X_test_sm = sm.add_constant(X_test, has_constant='add')

# Baseline predictions (mean booking rate)
y_pred_baseline = np.repeat(y_train.mean(), len(y_test))

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)
print('Baseline mean booking rate:', y_train.mean())

## 3) Fit Linear Regression (OLS)

In [ ]:
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

y_pred = ols_model.predict(X_test_sm)

# Clip predictions to [0, 1] for probability interpretation
y_pred_clipped = np.clip(y_pred, 0, 1)

## 4) Evaluate Model Quality

We report RMSE, MAE, and R² for both baseline and OLS predictions.

In [ ]:
def regression_metrics(y_true, y_hat):
    return {
        'RMSE': mean_squared_error(y_true, y_hat, squared=False),
        'MAE': mean_absolute_error(y_true, y_hat),
        'R2': r2_score(y_true, y_hat),
    }

results = pd.DataFrame([
    {'Model': 'Baseline (mean)', **regression_metrics(y_test, y_pred_baseline)},
    {'Model': 'OLS (raw pred)', **regression_metrics(y_test, y_pred)},
    {'Model': 'OLS (clipped 0-1)', **regression_metrics(y_test, y_pred_clipped)},
])

results.sort_values('RMSE').reset_index(drop=True)

## 5) Coefficient Interpretation

Positive coefficients indicate higher predicted booking probability, holding other features fixed.

In [ ]:
coef_table = (
    pd.DataFrame({
        'feature': ols_model.params.index,
        'coef': ols_model.params.values,
        'pvalue': ols_model.pvalues.values,
    })
    .query("feature != 'const'")
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)

print('Top positive effects:')
display(coef_table.sort_values('coef', ascending=False).head(8))

print('Top negative effects:')
display(coef_table.sort_values('coef', ascending=True).head(8))

## 6) Residual Diagnostics

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(residuals, bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual (y_true - y_pred)')

axes[1].scatter(y_pred, residuals, s=8, alpha=0.3)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted booking probability (raw OLS)')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

print('Predictions outside [0,1]:', ((y_pred < 0) | (y_pred > 1)).mean())

## 7) Conclusions (Fill After Running)

Use this section in your write-up:

1. **Model performance:** Report RMSE/MAE/R² and compare against baseline.
2. **Most influential features:** Name 2-4 strongest positive/negative coefficients.
3. **Interpretation caveat:** Since outcome is binary, OLS is an approximation; logistic regression remains the primary probability model.
4. **Next step:** Compare signs/magnitudes with `q2_diagnostics.ipynb` logistic coefficients to check directional consistency.